In [2]:
# Import pip_system_certs BEFORE importing requests or urllib3
import pip_system_certs.wrapt_requests
pip_system_certs.wrapt_requests.inject_truststore()

# Then import other libraries
import requests
import urllib3

In [1]:
from fastcoref import FCoref

C:\Users\TharunramSreenivasan\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:

model = FCoref()

02/09/2026 13:59:52 - INFO - 	 HTTP Request: HEAD https://huggingface.co/biu-nlp/f-coref/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
02/09/2026 13:59:52 - INFO - 	 HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/biu-nlp/f-coref/e91dfff12879495d882ba9460d0c5d5dd44ade59/config.json "HTTP/1.1 200 OK"
02/09/2026 13:59:52 - INFO - 	 HTTP Request: HEAD https://huggingface.co/biu-nlp/f-coref/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
02/09/2026 13:59:52 - INFO - 	 HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/biu-nlp/f-coref/e91dfff12879495d882ba9460d0c5d5dd44ade59/config.json "HTTP/1.1 200 OK"
02/09/2026 13:59:52 - INFO - 	 HTTP Request: HEAD https://huggingface.co/biu-nlp/f-coref/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
02/09/2026 13:59:52 - INFO - 	 HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/biu-nlp/f-coref/e91dfff12879495d882ba9460d0c5d5dd44ade59/tokenizer_config.json

AttributeError: 'FCorefModel' object has no attribute 'all_tied_weights_keys'

02/09/2026 13:59:55 - INFO - 	 HTTP Request: GET https://huggingface.co/api/models/biu-nlp/f-coref/discussions?p=0 "HTTP/1.1 200 OK"
02/09/2026 13:59:55 - INFO - 	 HTTP Request: GET https://huggingface.co/api/models/biu-nlp/f-coref/commits/refs%2Fpr%2F1 "HTTP/1.1 200 OK"
02/09/2026 13:59:56 - INFO - 	 HTTP Request: HEAD https://huggingface.co/biu-nlp/f-coref/resolve/refs%2Fpr%2F1/model.safetensors.index.json "HTTP/1.1 404 Not Found"
02/09/2026 13:59:56 - INFO - 	 HTTP Request: HEAD https://huggingface.co/biu-nlp/f-coref/resolve/refs%2Fpr%2F1/model.safetensors "HTTP/1.1 302 Found"
02/09/2026 13:59:56 - INFO - 	 HTTP Request: GET https://huggingface.co/api/models/biu-nlp/f-coref/xet-read-token/d5a382c8bfe1105cee1a73007525ee08ab693d9a "HTTP/1.1 200 OK"


In [2]:
import spacy
from typing import List
from spacy.tokens import Doc, Span
from fastcoref import FCoref
 

nlp = spacy.load('en_core_web_sm')
model = FCoref()
 

C:\Users\TharunramSreenivasan\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
02/24/2026 17:08:22 - INFO - 	 missing_keys: []
02/24/2026 17:08:22 - INFO - 	 unexpected_keys: []
02/24/2026 17:08:22 - INFO - 	 mismatched_keys: []
02/24/2026 17:08:22 - INFO - 	 error_msgs: []
02/24/2026 17:08:22 - INFO - 	 Model Parameters: 90.5M, Transformer: 82.1M, Coref head: 8.4M


In [5]:

def core_logic_part(document: Doc, coref: List[int], resolved: List[str], mention_span: Span):
    final_token = document[coref[1]]
    if final_token.tag_ in ["PRP$", "POS"]:
        resolved[coref[0]] = mention_span.text + "'s" + final_token.whitespace_
    else:
        resolved[coref[0]] = mention_span.text + final_token.whitespace_
    for i in range(coref[0] + 1, coref[1] + 1):
        resolved[i] = ""
    return resolved

def get_span_noun_indices(doc: Doc, cluster: List[List[int]]) -> List[int]:
    spans = [doc[span[0]:span[1]+1] for span in cluster]
    spans_pos = [[token.pos_ for token in span] for span in spans]
    span_noun_indices = [i for i, span_pos in enumerate(spans_pos)
        if any(pos in span_pos for pos in ['NOUN', 'PROPN'])]
    return span_noun_indices

def get_cluster_head(doc: Doc, cluster: List[List[int]], noun_indices: List[int]):
    head_idx = noun_indices[0]
    head_start, head_end = cluster[head_idx]
    head_span = doc[head_start:head_end+1]
    return head_span, [head_start, head_end]

def is_containing_other_spans(span: List[int], all_spans: List[List[int]]):
    return any([s[0] >= span[0] and s[1] <= span[1] and s != span for s in all_spans])

def improved_replace_corefs(document, clusters):
    resolved = list(tok.text_with_ws for tok in document)
    all_spans = [span for cluster in clusters for span in cluster]  # flattened list of all spans

    for cluster in clusters:
        noun_indices = get_span_noun_indices(document, cluster)

        if noun_indices:
            mention_span, mention = get_cluster_head(document, cluster, noun_indices)

            for coref in cluster:
                if coref != mention and not is_containing_other_spans(coref, all_spans):
                    core_logic_part(document, coref, resolved, mention_span)

    return "".join(resolved)

def get_fast_cluster_spans(doc, clusters):
    fast_clusters = []
    for cluster in clusters:
        new_group = []
        for tuple in cluster:
            print(type(tuple), tuple)
            (start, end) = tuple
            print("start, end", start, end)
            span = doc.char_span(start, end)
            print('span', span.start, span.end)
            new_group.append([span.start, span.end-1])
        fast_clusters.append(new_group)
    return fast_clusters

def get_fastcoref_clusters(doc, text):
    preds = model.predict(texts=[text])
    fast_clusters = preds[0].get_clusters(as_strings=False)
    fast_cluster_spans = get_fast_cluster_spans(doc, fast_clusters)
    return fast_cluster_spans


In [6]:
text = "Review by Michael Wood. Yesterday I saw the movie Jaws. It was incredible. The movie left a lasting impression on me"

In [7]:
doc = nlp(text)
    #clusters = get_allennlp_clusters(text)
clusters = get_fastcoref_clusters(doc, text)
coref_text = improved_replace_corefs(doc, clusters)

02/09/2026 14:10:53 - INFO - 	 Tokenize 1 inputs...


Map: 100%|██████████| 1/1 [00:00<00:00, 200.64 examples/s]
02/09/2026 14:10:53 - INFO - 	 ***** Running Inference on 1 texts *****
Inference: 100%|██████████| 1/1 [00:00<00:00, 44.96it/s]

<class 'tuple'> (10, 22)
start, end 10 22
span 2 4
<class 'tuple'> (34, 35)
start, end 34 35
span 6 7
<class 'tuple'> (114, 116)
start, end 114 116
span 23 24
<class 'tuple'> (40, 54)
start, end 40 54
span 8 11
<class 'tuple'> (56, 58)
start, end 56 58
span 12 13
<class 'tuple'> (75, 84)
start, end 75 84
span 16 18


In [8]:
from fastcoref import FCoref

# 1. Load the model directly
model = FCoref(model_name_or_path='biu-nlp/f-coref', device='cpu')

# 2. Define your text
text = "Elon Musk is the CEO of Tesla. He founded it many years ago."

# 3. Predict (returns a list of CorefResult objects)
preds = model.predict(texts=[text])

# 4. Get the resolved text for the first document
# This replaces pronouns with the original entity name
resolved_text = preds[0].get_resolved_content()

print(f"Original: {text}")
print(f"Resolved: {resolved_text}")
# Output: "Elon Musk is the CEO of Tesla. Elon Musk founded Tesla many years ago."


02/09/2026 14:12:21 - INFO - 	 missing_keys: []
02/09/2026 14:12:21 - INFO - 	 unexpected_keys: []
02/09/2026 14:12:21 - INFO - 	 mismatched_keys: []
02/09/2026 14:12:21 - INFO - 	 error_msgs: []
02/09/2026 14:12:21 - INFO - 	 Model Parameters: 90.5M, Transformer: 82.1M, Coref head: 8.4M
02/09/2026 14:12:21 - INFO - 	 Tokenize 1 inputs...
Map: 100%|██████████| 1/1 [00:00<00:00, 273.80 examples/s]
02/09/2026 14:12:22 - INFO - 	 ***** Running Inference on 1 texts *****
Inference: 100%|██████████| 1/1 [00:00<00:00, 52.00it/s]


AttributeError: 'CorefResult' object has no attribute 'get_resolved_content'

In [10]:
from fastcoref import FCoref
model = FCoref()

02/09/2026 14:15:40 - INFO - 	 missing_keys: []
02/09/2026 14:15:40 - INFO - 	 unexpected_keys: []
02/09/2026 14:15:40 - INFO - 	 mismatched_keys: []
02/09/2026 14:15:40 - INFO - 	 error_msgs: []
02/09/2026 14:15:40 - INFO - 	 Model Parameters: 90.5M, Transformer: 82.1M, Coref head: 8.4M


In [11]:
# The email text
email_text = [
    "I'm writing to file a claim regarding an accident involving my car. Yesterday, while I was driving on Main Street, another vehicle collided with mine. The driver of the other car admitted fault at the scene. He gave me his insurance details, and I have attached them to this email. My car suffered significant damage, and I sustained minor injuries. Please let me know the next steps in this process."
]

# Predict coreferences
preds = model.predict(texts=email_text)

# Get coreference clusters
clusters = preds[0].get_clusters()
print("Coreference Clusters:", clusters)

02/09/2026 14:15:56 - INFO - 	 Tokenize 1 inputs...
Map: 100%|██████████| 1/1 [00:00<00:00, 111.13 examples/s]
02/09/2026 14:15:56 - INFO - 	 ***** Running Inference on 1 texts *****
Inference: 100%|██████████| 1/1 [00:00<00:00, 27.52it/s]

Coreference Clusters: [['I', 'my', 'I', 'mine', 'me', 'I', 'My', 'I', 'me'], ['The driver of the other car', 'He', 'his'], ['his insurance details', 'them'], ['my car', 'My car']]


In [12]:
# Original email text
email_text = [
    "I'm writing to file a claim regarding an accident involving my car. Yesterday, while I was driving on Main Street, another vehicle collided with mine. The driver of the other car admitted fault at the scene. He gave me his insurance details, and I have attached them to this email. My car suffered significant damage, and I sustained minor injuries. Please let me know the next steps in this process."
]

# Coreference resolution
preds = model.predict(texts=email_text)
clusters = preds[0].get_clusters()

# Replace coreferences with entities
resolved_text = email_text[0]
for cluster in clusters:
    # Choose the entity for replacement (e.g., the first mention in the cluster)
    entity = cluster[0]
    for token in cluster:
        # Replace each token in the cluster with the chosen entity
        resolved_text = resolved_text.replace(token, entity)

# Print resolved text
print("Resolved Text:")
print(resolved_text)

02/09/2026 14:16:13 - INFO - 	 Tokenize 1 inputs...
Map: 100%|██████████| 1/1 [00:00<00:00, 131.07 examples/s]
02/09/2026 14:16:13 - INFO - 	 ***** Running Inference on 1 texts *****
Inference: 100%|██████████| 1/1 [00:00<00:00, 29.23it/s]

Resolved Text:
I'm writing to file a claim regarding an accident involving I car. Yesterday, while I was driving on Main Street, another vehicle collided with I. The driver of the other car admitted fault at the scene. The driver of the other car gave I The driver of the other car insurance details, and I have attached his insurance details to tThe driver of the other car email. I car suffered significant damage, and I sustained minor injuries. Please let I know the next steps in tThe driver of the other car process.


In [14]:
# Original email text
email_text = [
    "I'm writing to file a claim regarding an accident involving my car. Yesterday, while I was driving on Main Street, another vehicle collided with mine. The driver of the other car admitted fault at the scene. He gave me his insurance details, and I have attached them to this email. My car suffered significant damage, and I sustained minor injuries. Please let me know the next steps in this process."
]
email_text = ["Alice went to the park. She saw a bird. It was blue."]
# Coreference resolution
preds = model.predict(texts=email_text)
clusters = preds[0].get_clusters()

# Replace coreferences with entities
resolved_text = email_text[0]
for cluster in clusters:
    # Choose the entity for replacement (e.g., the first mention in the cluster)
    entity = cluster[0]
    for token in cluster:
        # Replace each token in the cluster with the chosen entity
        resolved_text = resolved_text.replace(token, entity)

# Print resolved text
print("Resolved Text:")
print(resolved_text)

02/09/2026 14:20:59 - INFO - 	 Tokenize 1 inputs...
Map: 100%|██████████| 1/1 [00:00<00:00, 170.58 examples/s]
02/09/2026 14:20:59 - INFO - 	 ***** Running Inference on 1 texts *****
Inference: 100%|██████████| 1/1 [00:00<00:00, 46.97it/s]

Resolved Text:
Alice went to the park. She saw a bird. a bird was blue.


In [3]:
from fastcoref import LingMessCoref

model_l = LingMessCoref()
 

02/24/2026 17:08:55 - INFO - 	 missing_keys: []
02/24/2026 17:08:55 - INFO - 	 unexpected_keys: []
02/24/2026 17:08:55 - INFO - 	 mismatched_keys: []
02/24/2026 17:08:55 - INFO - 	 error_msgs: []
02/24/2026 17:08:55 - INFO - 	 Model Parameters: 590.0M, Transformer: 434.6M, Coref head: 155.4M


In [3]:
from fastcoref import FCoref

model = FCoref()
email_text = ["Alice is a carpenter. She went to the shed." ]
preds = model.predict(texts=email_text)

# Get clusters with character indices (as_strings=False)
clusters = preds[0].get_clusters(as_strings=False)
text = email_text[0]

# 1. Create a list of all mentions to replace, excluding the first one (the representative)
replacements = []
for cluster in clusters:
    representative = text[cluster[0][0]:cluster[0][1]] # The first mention (e.g., "Alice")
    for i in range(1, len(cluster)): # Start from the second mention
        start, end = cluster[i]
        replacements.append((start, end, representative))

# 2. Sort replacements in REVERSE order (from end of string to beginning)
# This prevents index shifting as the string length changes
replacements.sort(key=lambda x: x[0], reverse=True)

# 3. Perform the replacement
resolved_text = text
for start, end, rep in replacements:
    resolved_text = resolved_text[:start] + rep + resolved_text[end:]

print("Correctly Resolved Text:")
print(resolved_text)


02/09/2026 14:40:44 - INFO - 	 missing_keys: []
02/09/2026 14:40:44 - INFO - 	 unexpected_keys: []
02/09/2026 14:40:44 - INFO - 	 mismatched_keys: []
02/09/2026 14:40:44 - INFO - 	 error_msgs: []
02/09/2026 14:40:44 - INFO - 	 Model Parameters: 90.5M, Transformer: 82.1M, Coref head: 8.4M
02/09/2026 14:40:44 - INFO - 	 Tokenize 1 inputs...
Map: 100%|██████████| 1/1 [00:00<00:00, 153.71 examples/s]
02/09/2026 14:40:44 - INFO - 	 ***** Running Inference on 1 texts *****
Inference: 100%|██████████| 1/1 [00:00<00:00, 56.04it/s]

Correctly Resolved Text:
Alice is a carpenter. Alice went to the shed.


In [4]:
email_text = ["obama is the 44th president , he was born in thailand" ]
preds = model.predict(texts=email_text)

# Get clusters with character indices (as_strings=False)
clusters = preds[0].get_clusters(as_strings=False)
text = email_text[0]

# 1. Create a list of all mentions to replace, excluding the first one (the representative)
replacements = []
for cluster in clusters:
    representative = text[cluster[0][0]:cluster[0][1]] # The first mention (e.g., "Alice")
    for i in range(1, len(cluster)): # Start from the second mention
        start, end = cluster[i]
        replacements.append((start, end, representative))

# 2. Sort replacements in REVERSE order (from end of string to beginning)
# This prevents index shifting as the string length changes
replacements.sort(key=lambda x: x[0], reverse=True)

# 3. Perform the replacement
resolved_text = text
for start, end, rep in replacements:
    resolved_text = resolved_text[:start] + rep + resolved_text[end:]

print("Correctly Resolved Text:")
print(resolved_text)


02/09/2026 14:46:39 - INFO - 	 Tokenize 1 inputs...
Map: 100%|██████████| 1/1 [00:00<00:00, 160.07 examples/s]
02/09/2026 14:46:39 - INFO - 	 ***** Running Inference on 1 texts *****
Inference: 100%|██████████| 1/1 [00:00<00:00, 63.34it/s]

Correctly Resolved Text:
obama is the 44th president , obama was born in thailand


In [6]:
! pip install nltk

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ----------------------

In [1]:
from fastcoref import FCoref
# Load model once to save memory
model = FCoref()

def resolve_context(text):
    """Step 1: Replace pronouns with fully specified entities."""
    preds = model.predict(texts=[text])
    clusters = preds[0].get_clusters(as_strings=False)
    
    # Process replacements from back to front to maintain index integrity
    replacements = []
    for cluster in clusters:
        representative = text[cluster[0][0]:cluster[0][1]]
        for start, end in cluster[1:]:
            replacements.append((start, end, representative))
    
    replacements.sort(key=lambda x: x[0], reverse=True)
    
    resolved_text = text
    for start, end, rep in replacements:
        resolved_text = resolved_text[:start] + rep + resolved_text[end:]
    return resolved_text


C:\Users\TharunramSreenivasan\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\TharunramSreenivasan\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
02/09/2026 15:53:12 - INFO - 	 missing_keys: []
02/09/2026 15:53:12 - INFO - 	 unexpected_keys: []
02/09/2026 15:53:12 - INFO - 	 mismatched_keys: []
02/09/2026 15:53:12 - INFO - 	 error_msgs: []
02/09/2026 15:53:12 - INFO - 	 Model Parameters:

In [ ]:

from fastcoref import LingMessCoref
model = LingMessCoref(device='cpu')

C:\Users\TharunramSreenivasan\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [2]:
import nltk
from nltk.tokenize import sent_tokenize

In [7]:
import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\TharunramSreenivasan\AppData\Roaming\nltk_dat
[nltk_data]     a...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


True

In [3]:
def get_atomic_units(resolved_docs):
    """Step 2: Split documents into a sequence of sentences."""
    all_sentences = []
    for doc in resolved_docs:
        # Using NLTK to handle complex punctuation
        sentences = sent_tokenize(doc)
        all_sentences.extend(sentences)
    return all_sentences


In [4]:
def remove_near_duplicates(sentences, threshold=0.9):
    """Step 3: Discard sentences with Jaccard similarity >= threshold."""
    unique_sentences = []
    
    for current_s in sentences:
        # BoW Representation
        words_current = set(current_s.lower().split())
        is_duplicate = False
        
        for seen_s in unique_sentences:
            words_seen = set(seen_s.lower().split())
            
            # Calculate Jaccard Similarity
            intersection = len(words_current.intersection(words_seen))
            union = len(words_current.union(words_seen))
            similarity = intersection / union if union > 0 else 0
            
            if similarity >= threshold:
                is_duplicate = True
                break
        
        if not is_duplicate:
            unique_sentences.append(current_s)
            
    return unique_sentences


In [5]:
def resolve_context_smart(text):
    preds = model.predict(texts=[text])
    clusters = preds.get_clusters(as_strings=False)
    
    replacements = []
    for cluster in clusters:
        # STRATEGY: Pick the "best" representative (longest or most specific)
        # Instead of cluster[0], find the one that is likely a Proper Noun
        mentions = [text[start:end] for start, end in cluster]
        representative = max(mentions, key=len) # Simplest: pick the longest string
        
        for start, end in cluster:
            # Only replace if the mention isn't already the representative
            if text[start:end] != representative:
                replacements.append((start, end, representative))
    
    replacements.sort(key=lambda x: x[0], reverse=True)
    
    resolved_text = text
    for start, end, rep in replacements:
        resolved_text = resolved_text[:start] + rep + resolved_text[end:]
    return resolved_text


In [6]:
file_path = 'text.txt'

try:
    with open(file_path, 'r', encoding='utf-8') as file:
        content = file.read()
except FileNotFoundError:
    print(f"Error: The file '{file_path}' was not found.")
except Exception as e:
    print(f"An error occurred: {e}")


In [7]:
type(content)

str

In [12]:
test_docs = [
    "Alice is a researcher at the lab. The researcher found a breakthrough. She published it in Nature.",
    "The algorithm converges in ten iterations.",
    "The algorithm converges in 10 iterations."
]

#test_docs = ["Text - The customer is trying to use the Outlook client on their new multi-user or multi-session server. While trying to send an email from a test bot using the Email package (Send command), a Microsoft Outlook pop-up appears saying:   &quot;A program is trying to send an email message on your behalf. If this is unexpected, click Deny and verify your antivirus software is up-to-date.&quot;    The cause:  This warning message is displayed when a program (A360 or others) tries to access the Outlook client to send an email message on behalf of the user, and your antivirus software is detected to be inactive or out-of-date.Refer the customer/user to this Microsoft Outlook article and have them review and decide on which resolution the will use based on their security requirements and configuration:  https://learn.microsoft.com/en-us/outlook/troubleshoot/security/a-program-is-trying-to-send-an-email-message-on-your-behalfThis error is can be triggered by other applications/tools, such as UI Path, as well."]
 
resolved = [resolve_context(d) for d in test_docs]
units = get_atomic_units(resolved)
final = remove_near_duplicates(units, threshold=0.7)  

 
with open("output.txt", "w", encoding="utf-8") as file:
    for s in final:
         
        file.write(f"{s}\n")

02/09/2026 16:13:08 - INFO - 	 Tokenize 1 inputs...
Map: 100%|██████████| 1/1 [00:00<00:00, 58.88 examples/s]
02/09/2026 16:13:09 - INFO - 	 ***** Running Inference on 1 texts *****
Inference: 100%|██████████| 1/1 [00:00<00:00, 10.57it/s]
02/09/2026 16:13:09 - INFO - 	 Tokenize 1 inputs...
Map: 100%|██████████| 1/1 [00:00<00:00, 90.70 examples/s]
02/09/2026 16:13:09 - INFO - 	 ***** Running Inference on 1 texts *****
Inference: 100%|██████████| 1/1 [00:00<00:00, 16.15it/s]
02/09/2026 16:13:09 - INFO - 	 Tokenize 1 inputs...
Map: 100%|██████████| 1/1 [00:00<00:00, 99.45 examples/s]
02/09/2026 16:13:09 - INFO - 	 ***** Running Inference on 1 texts *****
Inference: 100%|██████████| 1/1 [00:00<00:00, 22.64it/s]


In [1]:
from fastcoref import LingMessCoref
model = LingMessCoref()

C:\Users\TharunramSreenivasan\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\TharunramSreenivasan\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
C:\Users\TharunramSreenivasan\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses sym

In [6]:
text = "John went to the store. He bought some groceries. The man was happy with his purchase."

result = model.predict(text)

# Get the resolved text (this is what you mainly want)
resolved_text = result.text
print("Resolved text:", resolved_text)

# For clusters, let's inspect the structure first
print("\nCluster structure analysis:")
for i, cluster in enumerate(result.clusters):
    print(f"Cluster {i+1}:")
    print(f"  Raw cluster: {cluster}")
    print(f"  Type of first item: {type(cluster[0]) if cluster else 'Empty'}")
    if cluster and hasattr(cluster[0], '__len__') and len(cluster[0]) > 0:
        print(f"  First item content: {cluster[0]}")
        print(f"  Types in first item: {[type(x) for x in cluster[0]]}")

02/10/2026 11:01:36 - INFO - 	 Tokenize 1 inputs...
Map: 100%|██████████| 1/1 [00:00<00:00, 65.24 examples/s]
02/10/2026 11:01:36 - INFO - 	 ***** Running Inference on 1 texts *****
Inference: 100%|██████████| 1/1 [00:01<00:00,  1.58s/it]

Resolved text: John went to the store. He bought some groceries. The man was happy with his purchase.

Cluster structure analysis:
Cluster 1:
  Raw cluster: ((np.int64(1), np.int64(1)), (np.int64(7), np.int64(7)), (np.int64(12), np.int64(13)), (np.int64(17), np.int64(17)))
  Type of first item: <class 'tuple'>
  First item content: (np.int64(1), np.int64(1))
  Types in first item: [<class 'numpy.int64'>, <class 'numpy.int64'>]


In [12]:
# Test with more obvious coreference examples
test_docs = [
    "Sarah is a doctor. She works at the hospital. The woman is very dedicated.",
    "Microsoft released a new product. The company expects high sales. It will be available next month.",
    "Tom bought a car. He loves the vehicle. The man drives it every day.",
    "The book was on the table. It was very interesting. The novel had 300 pages."
]


In [20]:
test_docs = [
    "Barack Obama was president. He served two terms. The politician is now retired.",
    "Microsoft released Windows. The company expects good sales. It will be popular.",
    "The book was interesting. It had 300 pages. The novel was well-written."
]

In [57]:

def resolve_context(text):
    """Step 1: Replace pronouns with fully specified entities."""
    preds = model.predict(texts=[text])
    clusters = preds[0].get_clusters(as_strings=False)
    
    # Process replacements from back to front to maintain index integrity
    replacements = []
    for cluster in clusters:
        representative = text[cluster[0][0]:cluster[0][1]]
        for start, end in cluster[1:]:
            replacements.append((start, end, representative))
    print(f"replacements1{replacements}")
    replacements.sort(key=lambda x: x[0], reverse=True)
    print(f"replacements2{replacements}")
    resolved_text = text
    for start, end, rep in replacements:
        resolved_text = resolved_text[:start] + rep + resolved_text[end:]
    return resolved_text

In [58]:
resolve_context("Barack Obama was president. He served two terms. The politician is now retired.")

02/10/2026 11:35:55 - INFO - 	 Tokenize 1 inputs...
Map: 100%|██████████| 1/1 [00:00<00:00, 152.85 examples/s]
02/10/2026 11:35:55 - INFO - 	 ***** Running Inference on 1 texts *****
Inference: 100%|██████████| 1/1 [00:01<00:00,  1.69s/it]

replacements1[(28, 30, 'Barack Obama'), (49, 63, 'Barack Obama')]
replacements2[(49, 63, 'Barack Obama'), (28, 30, 'Barack Obama')]


'Barack Obama was president. Barack Obama served two terms. Barack Obama is now retired.'

In [38]:
preds = model.predict(["Barack Obama was president. He served two terms. The politician is now retired."])

02/10/2026 11:25:00 - INFO - 	 Tokenize 1 inputs...
Map: 100%|██████████| 1/1 [00:00<00:00, 186.45 examples/s]
02/10/2026 11:25:00 - INFO - 	 ***** Running Inference on 1 texts *****
Inference: 100%|██████████| 1/1 [00:01<00:00,  1.58s/it]


In [41]:
preds[0].get_clusters()

[['Barack Obama', 'He', 'The politician']]

In [44]:
clusters = preds[0].get_clusters(as_strings = False)

In [4]:
import spacy
from fastcoref import spacy_component
 
nlp = spacy.load("en_core_web_sm", exclude=["parser", "lemmatizer", "ner", "textcat"])
 
nlp.add_pipe(
    "fastcoref",
    config={
        'model_architecture': 'LingMessCoref',
        'model_path': 'biu-nlp/lingmess-coref',
        'device': 'cpu'  
    }
)



02/24/2026 17:09:25 - INFO - 	 missing_keys: []
02/24/2026 17:09:25 - INFO - 	 unexpected_keys: []
02/24/2026 17:09:25 - INFO - 	 mismatched_keys: []
02/24/2026 17:09:25 - INFO - 	 error_msgs: []
02/24/2026 17:09:25 - INFO - 	 Model Parameters: 590.0M, Transformer: 434.6M, Coref head: 155.4M


In [6]:
nlp.add_pipe("sentencizer")

In [12]:
text = """When we try to connect/authenticate to Azure using Microsoft 365 Outlook package, the following errors could appear:    ERROR #1:  

  groupemail@domainname.com logged in with a different account. Please completely log out currently logged-in user from the browser and log in with the groupemail@domainname.com again      

    ERROR #2:  

  Unable to connect to the Microsoft 36S outlook server. Error Request_ResourceNotFound Resource com&#39;xxxx@xxx.com&#39; does not exist or one of Its queried reference-property objects are not present.  

    

    ERROR #3:  

  Unable to connect to the Microsoft 365 outlook server. Error :Authorization_RequestDenied Insufficient privileges to complete the operation.          

  The permissions from Azure side were validated and all were fine.  User profile read permission requirement. This single cause impacts differently depending of the scenario:   FOR ERRORS 1 AND 2:  The user principle name and email was different in Azure Active Directory for the User. 
 The Scope of the OAuth is incorrect.   FOR ERROR 3:  The Microsoft Graph application permission  User.Read.All  was not provided in the Azure App.    SOLUTION:   
   
 1) Update the Microsoft 365 Outlook package version to 1.2.3 or higher. Starting from this version, the user profile read permission is not longer needed.  2) Ensure the whitelisting/exclusion to our software, as well as to our URL&#39;s, IP&#39;s and ports are already implemented according to our requirements, being extremely careful to exclude &quot;  *graph.microsoft.com*  &quot; as indicated in the following article, under &quot;Additional information&quot;: https://apeople.automationanywhere.com/s/article/A2019-How-to-add-Bot-Agent-into-Anti-Ware-exception-list 
       WORKAROUND:   If you don&#39;t want to update Microsoft 365 Outlook package, or if the update did not work, try the following:      FOR ERRORS 1 AND 2:    
  Check the Scope used in the OAuth connection in the control room and update it to :  offline_access,openid, https://graph.microsoft.com/.default     Make sure we have both user principle name and email same:    
    
     FOR ERROR 3:    1)  Go to  API permissions .   
    
  2)  Click on  Add a permission .  3)  Click on  Microsoft API&#39;s .  4)  Select  Microsoft Graph .   
    
  5)  Select  Application permissions .   
    
  6)  Click on  Application permissions  and search for User.Read.All, check it and click on  Add permissions.    
    
   NOTE:  Ensure these  Application permissions  are included: 
  User.Read.All  Mail.ReadWrite  Mail.Send  
  7) Grant admin consent. Admin consent can be granted only by an Azure admin.  NOTES:   1) Microsoft 365 Outlook package version 1.2.3 can be found attached to this article (for On-Prem installations).  2) Check the latest version of Microsoft 365 Outlook package here: https://docs.automationanywhere.com/bundle/enterprise-v2019/page/enterprise-cloud/topics/aae-client/bot-creator/commands/micosoft-365-outlook-package-releases.html  3) Download the latest version of Microsoft 365 Outlook package from here (for On-Prem installations): https://automationanywhere.app.box.com/s/4qr9kgfsd84pzz9fem7troz0ec9z6w7h  4) Please make sure we add all the necessary permission in Azure side. The below documentation if for Microsoft Graph Setup OAuth 2.0:  
  4.1) Set up OAuth 2.0 using Client Credentials:  https://apeople.automationanywhere.com/s/article/How-to-setup-an-App-in-Azure-to-use-OAuth2-with-Client-Credentials-Flow-for-Microsoft-365-Outlook   4.2) Set up OAuth 2.0 using Authorization Code with PKCE:  https://apeople.automationanywhere.com/s/article/How-to-setup-an-App-in-Azure-to-use-OAuth2-with-PKCE-Flow-for-Microsoft-365-Outlook   4.3) Set up and Configure OAuth 2.0 using Control Room managed:  

  https://docs.automationanywhere.com/bundle/enterprise-v2019/page/set-up-oauth2-application-in-the-microsoft-azure-portal.html  https://docs.automationanywhere.com/bundle/enterprise-v2019/page/configure-oauth2-parameters-control-room.html  

"""

In [13]:
#text = "John sold his car. Alice was not happy with john she wanted to invest in some other place.He was happy. "
doc = nlp(text, component_cfg={"fastcoref": {'resolve_text': True}})

# Get the resolved text
resolved_text = doc._.resolved_text

# To get a list of resolved sentences:
# We process the resolved_text again so the sentencizer sees the new nouns
resolved_doc = nlp(resolved_text)
sentences = [sent.text for sent in resolved_doc.sents]

02/24/2026 17:15:02 - INFO - 	 Tokenize 1 inputs...
Map: 100%|██████████| 1/1 [00:00<00:00, 34.46 examples/s]
02/24/2026 17:15:02 - INFO - 	 ***** Running Inference on 1 texts *****
Inference: 100%|██████████| 1/1 [00:05<00:00,  5.21s/it]
02/24/2026 17:15:07 - INFO - 	 Tokenize 1 inputs...
Map: 100%|██████████| 1/1 [00:00<00:00, 29.30 examples/s]
02/24/2026 17:15:07 - INFO - 	 ***** Running Inference on 1 texts *****
Inference: 100%|██████████| 1/1 [00:04<00:00,  4.82s/it]


In [14]:
sentences

['When we try to connect/authenticate to Azure using Microsoft 365 Outlook package, the following errors could appear:    ERROR #1:  \n\n  groupemail@domainname.com logged in with a different account.',
 'Please completely log out a different account from the browser and log in with groupemail@domainname.com again      \n\n    ERROR #2:  \n\n  Unable to connect to the Microsoft 36S outlook server.',
 'Error Request_ResourceNotFound Resource com&#39;xxxx@xxx.com&#39; does not exist or one of Its queried reference-property objects are not present.',
 ' \n\n\n\n    ERROR #3:  \n\n  Unable to connect to the Microsoft 365 outlook server.',
 'Error :Authorization_RequestDenied Insufficient privileges to complete the operation.',
 '         \n\n  The permissions from Azure side were validated and all were fine.',
 ' User profile read permission requirement.',
 'This single cause impacts differently depending of the scenario:   FOR ERRORS 1 AND 2:  The user principle name and email was differe

In [92]:
def get_jaccard_sim(str1, str2):
    """Calculates Jaccard Similarity between two strings."""
    a = set(str1.lower().split()) 
    b = set(str2.lower().split())
    if not a or not b: return 0.0
    
    intersection = len(a.intersection(b))
    union = len(a.union(b))
    return intersection / union

def process_and_deduplicate(all_resolved_texts):
    unique_sentences = []
    
    for text in all_resolved_texts:
        # Split document into sentences
        doc = nlp(text)
        
        for sent in doc.sents:
            new_sent_text = sent.text.strip()
            if not new_sent_text: continue
            
            # Check similarity against sentences we already kept
            is_duplicate = False
            for existing_sent in unique_sentences:
                if get_jaccard_sim(new_sent_text, existing_sent) >= 0.9:
                    is_duplicate = True
                    break
            
            if not is_duplicate:
                unique_sentences.append(new_sent_text)
                
    return unique_sentences


In [94]:
dedp = process_and_deduplicate(sentences)

02/10/2026 12:52:39 - INFO - 	 Tokenize 1 inputs...
Map: 100%|██████████| 1/1 [00:00<00:00, 142.82 examples/s]
02/10/2026 12:52:39 - INFO - 	 ***** Running Inference on 1 texts *****
Inference: 100%|██████████| 1/1 [00:01<00:00,  1.17s/it]
02/10/2026 12:52:40 - INFO - 	 Tokenize 1 inputs...
Map: 100%|██████████| 1/1 [00:00<00:00, 108.82 examples/s]
02/10/2026 12:52:40 - INFO - 	 ***** Running Inference on 1 texts *****
Inference: 100%|██████████| 1/1 [00:01<00:00,  1.14s/it]
02/10/2026 12:52:41 - INFO - 	 Tokenize 1 inputs...
Map: 100%|██████████| 1/1 [00:00<00:00, 92.23 examples/s]
02/10/2026 12:52:41 - INFO - 	 ***** Running Inference on 1 texts *****
Inference: 100%|██████████| 1/1 [00:01<00:00,  1.15s/it]
02/10/2026 12:52:42 - INFO - 	 Tokenize 1 inputs...
Map: 100%|██████████| 1/1 [00:00<00:00, 144.98 examples/s]
02/10/2026 12:52:43 - INFO - 	 ***** Running Inference on 1 texts *****
Inference: 100%|██████████| 1/1 [00:01<00:00,  1.13s/it]
02/10/2026 12:52:44 - INFO - 	 Tokenize 1

In [96]:
len(dedp)

11

In [87]:
text = """When using Outlook as the configuration in any of the email commands, if the application is not installed properly, we get below error -  &quot; Either an Outlook account is not setup on your system or the Outlook application can not be found &quot; Potential causes of the error:
  MS Office 365 is not installed properly on the bot running device.  MS Office version that is installed is not compatible with MS Access Database Engine installed on the same machine.  One of the Registry Entries &quot; OUTLOOK.EXE &quot; inside &quot; Computer\HKEY_LOCAL_MACHINE\SOFTWARE\Microsoft\Windows\CurrentVersion\App Paths\ &quot; could be missing or pointing to an invalid path of the Outlook application.  Resolution Steps:
  To resolve this issue, we need to perform the repair of the Office application on the affected bot running device.  For Cause 2, during the repair process, we may get a prompt suggesting to switch the Office version from 64-bit to 32-bit (or vice-versa) based on the version of MS Access Database Engine installed. In such a case, we need to follow the suggestion to install the appropriate version of the Office application.  After the repair process completes successfully, reboot the device and then verify that the OUTLOOK.EXE Registry Entry mentioned is present in the specified path.  If the key is still missing, the repair did not happen properly and we may need to uninstall and re-install the Office application.  In case, the customer upgrade or downgrade the office version, there are registry entry of older version present. Go HKEY_CLASSES_ROOT\Interface\{00063001-0000-0000-C000-000000000046}\TypeLib and check the current version of outlook. Like below screenshot is having outlook version key 9.6 (2016 outlook)."""

<>:2: SyntaxWarning: invalid escape sequence '\H'
<>:2: SyntaxWarning: invalid escape sequence '\H'
C:\Users\TharunramSreenivasan\AppData\Local\Temp\ipykernel_35584\3397976315.py:2: SyntaxWarning: invalid escape sequence '\H'
  MS Office 365 is not installed properly on the bot running device.  MS Office version that is installed is not compatible with MS Access Database Engine installed on the same machine.  One of the Registry Entries &quot; OUTLOOK.EXE &quot; inside &quot; Computer\HKEY_LOCAL_MACHINE\SOFTWARE\Microsoft\Windows\CurrentVersion\App Paths\ &quot; could be missing or pointing to an invalid path of the Outlook application.  Resolution Steps:


In [83]:
#text = "Alice goes down the rabbit hole. She would discover a new reality."
doc = nlp(text, component_cfg={"fastcoref": {"resolve_text": True}})

# Access resolved text
print(doc._.resolved_text)
# Output: "Alice goes down the rabbit hole. Alice would discover a new reality."

# Access clusters
print(doc._.coref_clusters)


02/10/2026 12:08:33 - INFO - 	 Tokenize 1 inputs...


Map: 100%|██████████| 1/1 [00:00<00:00, 245.45 examples/s]
02/10/2026 12:08:33 - INFO - 	 ***** Running Inference on 1 texts *****
Inference: 100%|██████████| 1/1 [00:01<00:00,  1.36s/it]

When using Outlook as the configuration in any of the email commands, if Outlook is not installed properly, we get below error -  &quot; Either an Outlook account is not setup on your system or Outlook can not be found &quot; Potential causes of the error:
  MS Office 365 is not installed properly on the bot running device.  MS Office version that is installed is not compatible with MS Access Database Engine installed on the bot running device.  One of the Registry Entries &quot; OUTLOOK.EXE &quot; inside &quot; Computer\HKEY_LOCAL_MACHINE\SOFTWARE\Microsoft\Windows\CurrentVersion\App Paths\ &quot; could be missing or pointing to an invalid path of Outlook.  Resolution Steps:
  To resolve this issue, we need to perform the repair of the Office application on the bot running device.  For Cause 2, during the repair process, we may get a prompt suggesting to switch the Office version from 64-bit to 32-bit (or vice-versa) based on the version of MS Access Database Engine installed. In such

In [84]:
print(doc._.resolved_text)

When using Outlook as the configuration in any of the email commands, if Outlook is not installed properly, we get below error -  &quot; Either an Outlook account is not setup on your system or Outlook can not be found &quot; Potential causes of the error:
  MS Office 365 is not installed properly on the bot running device.  MS Office version that is installed is not compatible with MS Access Database Engine installed on the bot running device.  One of the Registry Entries &quot; OUTLOOK.EXE &quot; inside &quot; Computer\HKEY_LOCAL_MACHINE\SOFTWARE\Microsoft\Windows\CurrentVersion\App Paths\ &quot; could be missing or pointing to an invalid path of Outlook.  Resolution Steps:
  To resolve this issue, we need to perform the repair of the Office application on the bot running device.  For Cause 2, during the repair process, we may get a prompt suggesting to switch the Office version from 64-bit to 32-bit (or vice-versa) based on the version of MS Access Database Engine installed. In such

In [78]:
doc._.resolved_text

''

In [76]:
import spacy
nlp = spacy.load("en_core_web_sm")
doc = nlp(doc._.resolved_text)
sentences = [sent.text.strip() for sent in doc.sents]


In [77]:
sentences

[]

In [22]:
 
resolved = [resolve_context(d) for d in test_docs]

02/10/2026 11:17:47 - INFO - 	 Tokenize 1 inputs...
Map: 100%|██████████| 1/1 [00:00<00:00, 159.56 examples/s]
02/10/2026 11:17:47 - INFO - 	 ***** Running Inference on 1 texts *****
Inference: 100%|██████████| 1/1 [00:01<00:00,  1.56s/it]
02/10/2026 11:17:49 - INFO - 	 Tokenize 1 inputs...
Map: 100%|██████████| 1/1 [00:00<00:00, 75.97 examples/s]
02/10/2026 11:17:49 - INFO - 	 ***** Running Inference on 1 texts *****
Inference: 100%|██████████| 1/1 [00:01<00:00,  1.42s/it]
02/10/2026 11:17:50 - INFO - 	 Tokenize 1 inputs...
Map: 100%|██████████| 1/1 [00:00<00:00, 90.39 examples/s]
02/10/2026 11:17:51 - INFO - 	 ***** Running Inference on 1 texts *****
Inference: 100%|██████████| 1/1 [00:01<00:00,  1.44s/it]


In [23]:
resolved

['Barack Obama was president. Barack Obama served two terms. Barack Obama is now retired.',
 'Microsoft released Windows. Microsoft expects good sales. Windows will be popular.',
 'The book was interesting. The book had 300 pages. The book was well-written.']

In [ ]:
!pip install scikit-learn

In [100]:
from sklearn.feature_extraction.text import CountVectorizer
 

corpus = sentences

# Initialize and transform
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(corpus)

# Output word counts as an array
print(X.toarray())

# See the word to index mapping
print(vectorizer.vocabulary_)


[[0 0 0 ... 0 0 1]
 [0 0 0 ... 0 1 0]
 [0 0 0 ... 1 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [2 1 1 ... 0 0 0]
 [0 0 0 ... 0 0 0]]
{'when': 129, 'using': 123, 'outlook': 84, 'as': 17, 'the': 115, 'configuration': 33, 'in': 61, 'any': 12, 'of': 78, 'email': 44, 'commands': 29, 'if': 60, 'is': 67, 'not': 77, 'installed': 64, 'properly': 93, 'we': 128, 'get': 54, 'below': 20, 'error': 48, 'quot': 94, 'either': 43, 'an': 10, 'account': 8, 'setup': 103, 'on': 81, 'your': 132, 'system': 113, 'or': 83, 'can': 24, 'be': 19, 'found': 52, 'potential': 89, 'causes': 27, 'ms': 75, 'office': 79, '365': 5, 'bot': 22, 'running': 101, 'device': 39, 'version': 126, 'that': 114, 'compatible': 30, 'with': 131, 'access': 7, 'database': 38, 'engine': 45, 'one': 82, 'registry': 97, 'entries': 46, 'exe': 49, 'inside': 62, 'computer': 32, 'hkey_local_machine': 59, 'software': 104, 'microsoft': 73, 'windows': 130, 'currentversion': 36, 'app': 13, 'paths': 86, 'could': 34, 'missing': 74, 'pointing': 88, 'to': 119, 'invali

In [ ]:
from sklearn.metrics.pairwise import pairwise_distances

# 1. Convert BoW matrix to binary (presence/absence only)
# X is the output from your CountVectorizer.fit_transform()
X_binary = X.astype(bool).astype(int)

# 2. Calculate Jaccard Distance matrix
# 'jaccard' metric in sklearn expects binary input
jac_dist = pairwise_distances(X_binary.toarray(), metric='jaccard')

# 3. Convert Distance to Similarity
jac_sim = 1 - jac_dist

print(jac_sim)


In [102]:
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import jaccard_score

def deduplicate_sentences(sentences, threshold=0.9):
    if not sentences:
        return []

    # Start with the first sentence as our first unique entry
    unique_sentences = [sentences[0]]
    
    for i in range(1, len(sentences)):
        current_sentence = sentences[i]
        is_duplicate = False
        
        # Compare current sentence to all sentences already in our "unique" list
        for unique_sent in unique_sentences:
            # We must vectorize them together to ensure the same vocabulary/matrix shape
            vectorizer = CountVectorizer(binary=True).fit([current_sentence, unique_sent])
            vectors = vectorizer.transform([current_sentence, unique_sent]).toarray()
            
            # Calculate Jaccard Similarity
            similarity = jaccard_score(vectors[0], vectors[1])
            
            if similarity >= threshold:
                is_duplicate = True
                break
        
        if not is_duplicate:
            unique_sentences.append(current_sentence)
            
    return unique_sentences

# Example Usage
docs = [
    "The cat sits on the mat",
    "The cat is sitting on the mat", # High similarity
    "Data science is an interesting field",
    "The cat sits on a mat"          # Another near-duplicate
]

filtered_docs = deduplicate_sentences(docs, threshold=0.9)
print(f"Original count: {len(docs)} | Filtered count: {len(filtered_docs)}")


Original count: 4 | Filtered count: 3


In [17]:
def resolve_coreferences(document_text):
    """
    Apply coreference resolution to a single document and return clean text
    """
    result = model.predict(document_text)
    
    # Extract just the resolved text
    resolved_text = result.text
    
    return resolved_text


In [19]:
# Test with a very clear coreference example
document = "Sarah is a researcher. She works at the university. The scientist published a paper."

result = model.predict(document)

print("=== DEBUG INFO ===")
print("Original text:")
print(repr(document))
print("\nResolved text:")
print(repr(result.text))
print("\nAre they identical?", document == result.text)
print("\nClusters found:")
print(result.clusters)
print("Number of clusters:", len(result.clusters))


02/10/2026 11:16:11 - INFO - 	 Tokenize 1 inputs...
Map: 100%|██████████| 1/1 [00:00<00:00, 79.94 examples/s]
02/10/2026 11:16:11 - INFO - 	 ***** Running Inference on 1 texts *****
Inference: 100%|██████████| 1/1 [00:01<00:00,  1.59s/it]

=== DEBUG INFO ===
Original text:
'Sarah is a researcher. She works at the university. The scientist published a paper.'

Resolved text:
'Sarah is a researcher. She works at the university. The scientist published a paper.'

Are they identical? True

Clusters found:
[((np.int64(1), np.int64(1)), (np.int64(6), np.int64(6)), (np.int64(12), np.int64(13)))]
Number of clusters: 1
